# 14 — Latent Score-Based Reweighting

Import trực tiếp VAE/EDM/diffusion class tác giả; chuyển dữ liệu sang BAF và giới hạn ngân sách30/50 epoch. LSR dùng latent MLP cùng baseline none/balanced; có thêm transfer sang LightGBM với baseline riêng. Không gọi đây là tái lập nguyên xi các con số ICML2025. Nếu bỏ LSR từ11, notebook này in SKIPPED.

Không sửa cấu hình ở notebook này. Muốn đổi cấu hình hãy tạo campaign mới ở11.

In [1]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "src" / "baf.py").is_file():
    raise RuntimeError("Mở notebook từ thư mục BAF_project hoặc BAF_project/notebooks.")

CACHE_ROOT = Path(
    os.environ.get("BAF_CACHE_ROOT") or ROOT / ".cache" / "baf-research"
).expanduser().resolve()
for name, subdirectory in {
    "PIP_CACHE_DIR": "pip",
    "TORCH_HOME": "torch",
    "XDG_CACHE_HOME": "misc",
    "TEMP": "temp",
    "TMP": "temp",
}.items():
    folder = CACHE_ROOT / subdirectory
    folder.mkdir(parents=True, exist_ok=True)
    os.environ[name] = str(folder)

sys.path.insert(0, str(ROOT / "src"))
import gru

print("Python:", sys.executable)
print("Project:", ROOT)
print("Module:", gru.RESEARCH_VERSION)


Python: d:\UIT\.venvs\ml-gpu\Scripts\python.exe
Project: D:\UIT\DS317\BAF_project
Module: baf-research-20261028-v1


In [2]:
# Điền đường dẫn in ở notebook 11 nếu đã có nhiều campaign.
# Ví dụ: Path(r"D:\UIT\DS317\BAF_project\results\research_campaigns\weighting_research_...")
CAMPAIGN_DIR = None
CAMPAIGN = gru.research_resolve_campaign(ROOT, CAMPAIGN_DIR)
PLAN = gru.research_read(CAMPAIGN / "campaign.json")

print("Campaign:", CAMPAIGN)
print("Mode:", PLAN["mode"])
print("Suites:", PLAN["suites"])
print("Seeds:", PLAN["config"]["seeds"])


Campaign: D:\UIT\DS317\BAF_project\results\research_campaigns\weighting_research_20261028_214637_b9a2294e
Mode: full
Suites: ['losses', 'meta', 'lsr', 'xgboost']
Seeds: [42, 317, 2026]


## 1. Chạy training có notebook output thật

        Cell khởi chạy một Python process mới bằng đúng interpreter của kernel. Process thực thi từng cell trong notebook con, lưu stdout/display/error sau từng cell và in log trực tiếp bên dưới. Fit hoàn tất được kiểm hash rồi reuse khi resume. Nếu bị ngắt, không xóa campaign. Cell này có thể chạy lâu.

In [3]:
SUITE = 'lsr'
RUN_DIR = None
if SUITE in PLAN["suites"]:
    RUN_DIR = gru.research_launch_suite(CAMPAIGN, SUITE)
else:
    print("SKIPPED: suite không thuộc phạm vi đã chọn ở11:", SUITE)


BEGIN_STAGE 1
END_STAGE 1
BEGIN_STAGE 2
RESEARCH_STUDY lsr mode= full device= cuda
END_STAGE 2
BEGIN_STAGE 3
self.category_embeddings.weight.shape=torch.Size([31, 4])
LSR_VAE_EPOCH {'epoch': 1, 'train_loss': 0.8956457416248117, 'meta_reconstruction_ce': 0.004617565777152777, 'beta': 0.01}
LSR_VAE_EPOCH {'epoch': 2, 'train_loss': 0.04467734127712786, 'meta_reconstruction_ce': 0.0010677793761715293, 'beta': 0.01}
LSR_VAE_EPOCH {'epoch': 3, 'train_loss': 0.03627836374145823, 'meta_reconstruction_ce': 0.0006289445445872843, 'beta': 0.01}
LSR_VAE_EPOCH {'epoch': 4, 'train_loss': 0.0317695050001306, 'meta_reconstruction_ce': 0.0004961213562637568, 'beta': 0.01}
LSR_VAE_EPOCH {'epoch': 5, 'train_loss': 0.028211192454763223, 'meta_reconstruction_ce': 0.000221452399273403, 'beta': 0.01}
LSR_VAE_EPOCH {'epoch': 6, 'train_loss': 0.02567414501637756, 'meta_reconstruction_ce': 0.0003982357739005238, 'beta': 0.01}
LSR_VAE_EPOCH {'epoch': 7, 'train_loss': 0.02444338691595104, 'meta_reconstruction_ce'

## 2. Xem kết quả validation đã khóa

        Chưa có test metric ở notebook này. Model, configuration và threshold được lưu trước khi mở test ở16.

In [4]:
if RUN_DIR is not None:
    display(gru.research_suite_overview(RUN_DIR))
    print("Notebook training thật:")
    for executed in sorted((RUN_DIR / "attempts").glob("*/training_executed.ipynb")):
        print(executed)


,method,seed,validation_ap,threshold,fit_seconds,family,feature
0,lsr_lgbm_balanced,42,0.181084,0.803138,22.935875,lsr_lgbm,dense
1,lsr_lgbm_balanced,317,0.179549,0.798636,20.335903,lsr_lgbm,dense
2,lsr_lgbm_balanced,2026,0.179746,0.805836,24.038830,lsr_lgbm,dense
3,lsr_lgbm_none,42,0.180219,0.040145,20.012599,lsr_lgbm,dense
4,lsr_lgbm_none,317,0.179436,0.040668,21.159167,lsr_lgbm,dense
5,lsr_lgbm_none,2026,0.179119,0.040235,22.028915,lsr_lgbm,dense
6,lsr_lgbm_weighted,42,0.178763,0.043796,22.168257,lsr_lgbm,dense
7,lsr_lgbm_weighted,317,0.181462,0.043883,20.969631,lsr_lgbm,dense
8,lsr_lgbm_weighted,2026,0.178617,0.043931,21.433837,lsr_lgbm,dense
9,lsr_nn_balanced,42,0.170356,0.836052,233.108539,lsr_nn,latent


Notebook training thật:
D:\UIT\DS317\BAF_project\results\research_campaigns\weighting_research_20261028_214637_b9a2294e\suites\lsr\attempts\20261029_013935_dd364666\training_executed.ipynb


## 3. Xuất proof trong một lượt

        Không cần Ctrl+S hoặc chạy cell cuối hai lần. Full proof giữ trên máy; review nhỏ hơn để gửi, ghi rõ phần đã bỏ. Notebook `verify.ipynb` được execute thật, kiểm hash/score và không fit lại.

In [5]:
if RUN_DIR is not None:
    PROOF = gru.research_export_proof(RUN_DIR, ROOT)
    for kind, path in PROOF.items():
        print(kind, "→", path)


RESEARCH_FROZEN_SUITE_VERIFIED 18
FULL_PROOF D:\UIT\DS317\BAF_project\results\proofs\lsr_f791375a_full_proof.zip
PARTIAL_REVIEW D:\UIT\DS317\BAF_project\results\proofs\lsr_f791375a_review.zip
full_proof → D:\UIT\DS317\BAF_project\results\proofs\lsr_f791375a_full_proof.zip
review → D:\UIT\DS317\BAF_project\results\proofs\lsr_f791375a_review.zip
